In [ ]:
# RNN model

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf

import re
import random

In [4]:
# load the datasets

df = pd.read_csv("Day4_Dataset.csv")
df.head()

,review_id,review,sentiment
0,1,@filmfan21 The music was decent. The acting is...,negative
1,2,"<br />The villain was weak. A smart, funny and...",positive
2,3,Some scenes were too loud. The leading actors ...,positive
3,4,The first half was okay. It was not great. One...,negative
4,5,<br />The story is not good. The moive is base...,negative


In [5]:
df['review'][1]

'<br />The villain was weak. A smart, funny and touching film<br /><br />'

In [6]:
# clean
def clean(text):
    text = text.lower()                      # 1. lowercase
    text = re.sub(r"<.*?>", " ", text)       # 2. remove HTML tags
    text = re.sub(r"[^a-z ]", " ", text)     # 3. keep only letters and spaces
    return text
 
df["clean"] = df["review"].apply(clean)
df[["review", "clean"]].head(3)

,review,clean
0,@filmfan21 The music was decent. The acting is...,filmfan the music was decent the acting is...
1,"<br />The villain was weak. A smart, funny and...",the villain was weak a smart funny and touc...
2,Some scenes were too loud. The leading actors ...,some scenes were too loud the leading actors ...


In [7]:
# train_test_split
X = df["clean"]
y = (df["sentiment"] == "positive").astype(int)
 
X_train, X_test, y_train, y_test = train_test_split(
   X , y, test_size=0.2, random_state=42, stratify=y)
 
print("Training reviews:", len(X_train))
print("Test reviews    :", len(X_test))

Training reviews: 2400
Test reviews    : 600


In [8]:
# turn words into numbers(vectors)
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
 
vectorizer = layers.TextVectorization(max_tokens=5000, output_sequence_length=30)
vectorizer.adapt(X_train.values)             # learn the words of the training reviews
 
# a as s s s s da d asdas d as das  asd d ad(40) -> (
# my name is Rishav. -> (220,100,20,500,0,0,0,0,0,0,0,0,0,0,0,0,0,0,)
 
X_train_num = vectorizer(X_train.values).numpy()
X_test_num = vectorizer(X_test.values).numpy()
 
print("Review text :", X_train.iloc[0])
print("As numbers  :", X_train_num[0])

Review text : not bad at all  i did not expect to like it this much   
As numbers  : [  5  16  15  18   7  30   5 268  23 264   8  29  75   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0]


In [10]:
# RNN model 

keras.utils.set_random_seed(42)
 
rnn_model = keras.Sequential([
    keras.Input(shape=(30,)),
    layers.Embedding(5000, 16, mask_zero=True),
    layers.SimpleRNN(32),
    layers.Dense(1, activation="sigmoid"),
])
rnn_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
rnn_model.fit(X_train_num, y_train, epochs=10, verbose=0)
 
rnn_acc = rnn_model.evaluate(X_test_num, y_test, verbose=0)[1]
print("RNN test accuracy:", round(rnn_acc, 3))

RNN test accuracy: 0.862


In [13]:
def predict_review(text, model):
    # text = 'hello world'
    # model =  RNN MODEL

    number = vectorizer([clean(text)]).numpy()
            # [200,300,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]
    p = float(model.predict(number, verbose=0)[0][0])
            # model([200,300,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]) ->Sigmoid( 0 to 1)
    return ("positive" if p > 0.5 else "negative")

In [14]:
good = ["great", "lovely", "amazing", "brilliant"]
bad = ["terrible", "boring", "awful", "weak"]
 
random.seed(1)
texts, labels = [], []
for i in range(2000):
    g = random.choice(good)
    b = random.choice(bad)
    if random.random() < 0.5:
        texts.append(f"the acting was {g} but the ending was {b}")
        labels.append(0)                  # ends badly -> negative
    else:
        texts.append(f"the acting was {b} but the ending was {g}")
        labels.append(1)                  # ends well -> positive
 
for i in range(6):
    print(labels[i], "|", texts[i])

0 | the acting was lovely but the ending was terrible
0 | the acting was brilliant but the ending was weak
0 | the acting was brilliant but the ending was boring
0 | the acting was great but the ending was weak
0 | the acting was great but the ending was weak
1 | the acting was terrible but the ending was lovely


In [15]:
P_train, P_test, py_train, py_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

In [16]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
 
bow = CountVectorizer()
bow_model = LogisticRegression().fit(bow.fit_transform(P_train), py_train)
 
bow_acc = bow_model.score(bow.transform(P_test), py_test)
print("Bag of Words accuracy:", round(bow_acc, 3))

Bag of Words accuracy: 0.52


In [17]:
p_vectorizer = layers.TextVectorization(output_sequence_length=12)
p_vectorizer.adapt(np.array(P_train))
PX_train = p_vectorizer(np.array(P_train)).numpy()
PX_test = p_vectorizer(np.array(P_test)).numpy()
keras.utils.set_random_seed(42)

project_model = keras.Sequential([
    keras.Input(shape=(12,)),
    layers.Embedding(100, 16, mask_zero=True),
    layers.LSTM(32),
    layers.Dense(1, activation="sigmoid"),
])

project_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
project_model.fit(PX_train, np.array(py_train), epochs=15, verbose=0)
lstm_project_acc = project_model.evaluate(PX_test, np.array(py_test), verbose=0)[1]

print("LSTM accuracy:", round(lstm_project_acc, 3))

LSTM accuracy: 1.0
